In [13]:
import wilds
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from PIL import Image 
import torchvision.transforms as transforms 
import numpy as np
import pandas as pd

In [14]:
transform = transforms.Compose([transforms.PILToTensor()]) 

dataset = wilds.get_dataset("camelyon17",unlabeled=False, download=False)

In [15]:
dataset._source_domain_splits

AttributeError: 'Camelyon17Dataset' object has no attribute '_source_domain_splits'

In [27]:
a = dataset.get_input(0)
transform(a)

tensor([[[212, 208, 219,  ..., 221, 232, 221],
         [216, 214, 216,  ..., 226, 229, 218],
         [210, 217, 210,  ..., 223, 217, 212],
         ...,
         [188, 195, 198,  ..., 222, 220, 214],
         [188, 200, 208,  ..., 234, 233, 234],
         [195, 209, 216,  ..., 230, 232, 235]],

        [[150, 139, 144,  ..., 144, 151, 144],
         [141, 135, 138,  ..., 138, 141, 139],
         [130, 133, 136,  ..., 131, 137, 161],
         ...,
         [115, 125, 131,  ..., 191, 187, 194],
         [119, 136, 148,  ..., 219, 218, 210],
         [134, 155, 165,  ..., 214, 227, 218]],

        [[172, 169, 173,  ..., 179, 180, 175],
         [171, 166, 173,  ..., 167, 171, 174],
         [166, 165, 173,  ..., 161, 171, 187],
         ...,
         [154, 160, 165,  ..., 204, 201, 208],
         [157, 168, 176,  ..., 224, 223, 218],
         [169, 180, 186,  ..., 219, 227, 221]]], dtype=torch.uint8)

In [32]:
dataset._metadata_df['slide'].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49])

In [30]:
md_df.loc[md_df['patient']== '009']['tumor'].unique()

array([1, 0])

In [16]:
# make shorthand for usability
md_df = dataset._metadata_df

#save original index into variable so we can use to subset later
md_df['orig_idx'] = np.arange(md_df.shape[0])

#select only the samples from center 0
md_c0 = md_df.loc[md_df['center'] == 0]
idx_c0 = md_c0['orig_idx'].values

In [17]:
md_c0.shape

(59436, 9)

In [8]:
# declare empty tensor to store all images/labels in
X = torch.zeros((len(idx_c0),3, 96, 96), dtype = torch.uint8)
Y = torch.zeros((len(idx_c0)), dtype=torch.uint8)

# load samples from center 0 into array, 100 at a time
load_size = 100
remaining_idx = idx_c0
start = 0
end = load_size
while start < len(idx_c0):
    # to avoid errors at the end of the array
    end = min(end, len(idx_c0))
              
    # define the subset we want to load
    subset_idx = remaining_idx[start:end]

    # load the subset of images into the tensor
    # X[start:end, ...] = torch.stack([TF.to_tensor(dataset.get_input(idx)) for idx in subset_idx])
    T = torch.stack([TF.to_tensor(dataset.get_input(idx)) for idx in subset_idx])
    # retarget start and end
    start = end
    end += load_size

KeyboardInterrupt: 

In [20]:
(md_df.shape[0] * 3 * 96 * 96 * 8)/1000000000

100.849729536

In [47]:
#save original index into variable so we can use to subset later
md_df['orig_idx'] = np.arange(md_df.shape[0])

#select only the samples from center 0
md_c0 = md_df.loc[md_df['center'] == 0]
idx_c0 = md_c0['orig_idx'].values

subset_idx = idx_c0[0:100]
T = torch.stack([TF.to_tensor(dataset.get_input(i))for i in range(10)])
# T = torch.tensor([TF.to_tensor(dataset.get_input(idx)) for idx in subset_idx])


In [49]:
T.shape

torch.Size([10, 3, 96, 96])

In [41]:
md_c0['orig_idx']

0              0
1              1
2              2
3              3
4              4
           ...  
257690    257690
257691    257691
257692    257692
257693    257693
257694    257694
Name: orig_idx, Length: 59436, dtype: int64

In [40]:
md_c0.shape, md_df.shape

((59436, 9), (455954, 9))

In [30]:
dataset._input_array

['patches/patient_004_node_4/patch_patient_004_node_4_x_3328_y_21792.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_3200_y_22272.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_3168_y_22272.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_3328_y_21760.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_3232_y_22240.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_3168_y_22240.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_3136_y_22208.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_2656_y_18880.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_3136_y_22240.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_3296_y_21856.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_3296_y_21792.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_3360_y_21824.png',
 'patches/patient_004_node_4/patch_patient_004_node_4_x_3360_y_21760.png',
 'patches/patient_004_nod

In [8]:
dataset._metadata_df.shape

(455954, 8)

In [9]:
train_subset = dataset.get_subset("train")

In [17]:
train_subset.metadata_array.shape

torch.Size([302436, 4])

In [19]:
from wilds.common.data_loaders import get_train_loader

In [26]:
transform = transforms.Compose([transforms.PILToTensor()]) 

train_loader = get_train_loader("standard", train_subset, 1)

In [22]:
len(train_loader)

302436

In [42]:
class testNet(nn.Module):
    def __init__(self):
        super(testNet, self).__init__()

        self.c1 = nn.Conv2d(3, 3, 3)
        self.rl1 = nn.ReLU()
        self.c2 = nn.Conv2d(3, 1, 3)
        self.rl2 = nn.ReLU()
        self.fc1 = nn.Linear(8464, 2)

    def forward(self, x):
        x = self.c1(x)
        x = self.rl1(x)
        x = self.c2(x)
        x = self.rl2(x)
        x = torch.flatten(x)
        y = self.fc1(x)
        return(y)

    def train(self, trainloader, opt):
        crit = nn.CrossEntropyLoss()

        loss = 0
        for img, y_train, _ in trainloader:
            X_train = transform(img)

            out = self.forward(X_train.double())
            loss += crit(out, y_train)
        loss.backward()
        opt.step()

    def test(self, testloader):
        with torch.no_grad():
            total, correct = 0, 0
            for X_test, y_test in testloader:
                out = self.forward(X_test)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        accuracy = correct / total
        return accuracy

        

In [ ]:
net = testNet().double()
opt = torch.optim.SGD(net.parameters(), lr = 0.005)
net.train(train_subset, opt)

In [14]:
train_loader

In [21]:
for images, labels in train_loader:
    print(labels)

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'PIL.Image.Image'>